# Data checked into the policyengine-us-data repo

A number of public data sets have been directly checked into the policyengine-us-data repo
within the [storage](https://github.com/PolicyEngine/policyengine-us-data/tree/main/policyengine_us_data/storage)
directory. These can be seen by examining the storage directory, as seen below.

In [20]:
from policyengine_us_data.storage import STORAGE_FOLDER
from pathlib import Path

# Ensure STORAGE_FOLDER is a Path object
folder = Path(STORAGE_FOLDER)
print(f"The storage folder is at {STORAGE_FOLDER}\n")

print("Files in the storage folder with .csv or .h5 extensions, before any downloads:\n")
for item in folder.iterdir():
    if item.is_file() and item.suffix.lower() in ('.csv', '.h5'):
        print(item.name)

The storage folder is at /mnt/c/devl/policyengine-us-data/policyengine_us_data/storage

Files in the storage folder with .csv or .h5 extensions, before any downloads:

eitc.csv
healthcare_spending.csv
population_by_state.csv
spm_threshold_agi.csv
uprating_factors.csv
uprating_growth_factors.csv


## Data sets included in the repo
- eitc.csv: Earned Income Tax Credit
-  healthcare_spending.csv
- population_by_state.csv
- spm_threshold_agi.csv
- uprating_factors.csv
- uprating_growth_factors.csv

### eitc.csv: Earned Income Tax Credit Aggregate Returns

[Publication 1304](https://www.irs.gov/statistics/soi-tax-stats-individual-income-tax-returns-complete-report-publication-1304) is available for download from the IRS SOI as an entire pdf for each tax year. Below you can see where the `eitc.csv` data comes from for no children claimed.

![image](eitc_from_p1304.png)

# Downloads that happen at runtime

There are two programs with the sole pupose of downloading data from the internet,
which also live in the storage directory. These are:

- `download_public_prerequisites.py`, which downloads the following files from [Github assets](https://github.com/PolicyEngine/policyengine-us-data/releases) with the "release" tag ([see the json](https://api.github.com/repos/PolicyEngine/policyengine-us-data/releases/tags/release)), which is the tag for the original release in Aug 2014:
   * soi.csv (From the IRS SOI [Basic Tables](https://www.irs.gov/statistics/soi-tax-stats-individual-income-tax-returns-complete-report-publication-1304#Basic%20Tables))
   * np2023_d5_mid.csv
- `download_private_prerequisites.py`, which downloads the following files from Hugging Face Hub. These require the `HUGGING_FACE_TOKEN` to be set:
  * puf_2015.csv
  * demographics_2015.csv

In [21]:
%run ../policyengine_us_data/storage/download_public_prerequisites.py

100%|█████████████████████████████████████████████████████████████| 474k/474k [00:00<00:00, 6.92MB/s]
100%|███████████████████████████████████████████████████████████| 1.45M/1.45M [00:00<00:00, 10.6MB/s]


In [38]:
import os
os.environ["HUGGING_FACE_TOKEN"] = "<hugging face token>"

In [26]:

print(os.environ.get('HUGGING_FACE_TOKEN'))

<hugging face token>


In [27]:
%run ../policyengine_us_data/storage/download_private_prerequisites.py

# Datasets

The datasets section of the codebase lives in policyengine_us_data and contains a set of scripts that involves downloading, processing, and storing of the American Community Survey (ACS), the Current Population Survey (CPS), and the IRS Public Use File (PUF).

*TODO*: Sort out which data sets are coming from downloads and which are coming from h5.

## ACS

The ACS relies on a Dataset called CensusACS, which pulls from the following URS:
- spm_url = f"https://www2.census.gov/programs-surveys/supplemental-poverty-measure/datasets/spm/spm_{self.time_period}_pu.dta"
- person_url = f"https://www2.census.gov/programs-surveys/acs/data/pums/{self.time_period}/1-Year/csv_pus.zip"
- household_url = f"https://www2.census.gov/programs-surveys/acs/data/pums/{self.time_period}/1-Year/csv_hus.zip"

A Dataset class called CensusACS_2022 is constructed that has only light operations defined in its parent class
```
class CensusACS_2022(CensusACS):
    label = "Census ACS (2022)"
    name = "census_acs_2022.h5"
    file_path = STORAGE_FOLDER / "census_acs_2022.h5"
    time_period = 2022
    url = "hf://policyengine/policyengine-us-data/census_acs_2022.h5"
```

Inside the file housing these class definitions is the URL mapping to
the annual CPS data.
```
CPS_URL_BY_YEAR = {                                                                                                2018: "https://www2.census.gov/programs-surveys/cps/datasets/2019/march/asecpub19csv.zip",
    2019: "https://www2.census.gov/programs-surveys/cps/datasets/2020/march/asecpub20csv.zip",
    2020: "https://www2.census.gov/programs-surveys/cps/datasets/2021/march/asecpub21csv.zip",
    2021: "https://www2.census.gov/programs-surveys/cps/datasets/2022/march/asecpub22csv.zip",
    2022: "https://www2.census.gov/programs-surveys/cps/datasets/2023/march/asecpub23csv.zip",
    2023: "https://www2.census.gov/programs-surveys/cps/datasets/2024/march/asecpub24csv.zip",
}
```


Heavier operations are defined in the ACS class:

```
class ACS(Dataset):
    data_format = Dataset.ARRAYS
    time_period = None
    census_acs = None

    def generate(self) -> None:
        """Generates the ACS dataset."""

        raw_data = self.census_acs(require=True).load()
        acs = h5py.File(self.file_path, mode="w")
        person, household = [
            raw_data[entity] for entity in ("person", "household")
        ]

        self.add_id_variables(acs, person, household)
        self.add_person_variables(acs, person, household)
        self.add_household_variables(acs, household)
```
In the main loop, the generate function is called on a subclass of ACS:
```
class ACS_2022(ACS):
    name = "acs_2022"
    label = "ACS 2022"
    time_period = 2022
    file_path = STORAGE_FOLDER / "acs_2022.h5"
    census_acs = CensusACS_2022
    url = "release://PolicyEngine/policyengine-us-data/1.13.0/acs_2022.h5"
                                                                                                                                                                                          if __name__ == "__main__":
    ACS_2022().generate()
```


In [33]:
import policyengine_us_data.datasets.cps.cps as cps

In [34]:
cps.CPS_2021()

In [37]:
# A static method
#cps.CPS_2021().generate()
my_cps = cps.CPS_2021()
cps_df = my_cps.load_dataset()
cps_df


{'age': array([42, 76, 69, ..., 35, 70, 61]),
 'alimony_income': array([0, 0, 0, ..., 0, 0, 0]),
 'child_support_expense': array([0, 0, 0, ..., 0, 0, 0]),
 'child_support_received': array([0, 0, 0, ..., 0, 0, 0]),
 'county_fips': array([0, 0, 0, ..., 3, 3, 3]),
 'cps_race': array([ 1,  1,  1, ..., 21,  4,  8]),
 'disability_benefits': array([0, 0, 0, ..., 0, 0, 0]),
 'employment_income': array([ 44200,      0,  59000, ...,  57000,  45000, 100000]),
 'employment_income_last_year': array([ 4.5e+04, -1.0e+00,  6.0e+04, ..., -1.0e+00, -1.0e+00, -1.0e+00]),
 'family_id': array([    11,    191,    551, ..., 891731, 891821, 891901]),
 'farm_income': array([0, 0, 0, ..., 0, 0, 0]),
 'free_school_meals_reported': array([0, 0, 0, ..., 0, 0, 0]),
 'has_marketplace_health_coverage': array([ True, False, False, ..., False, False, False]),
 'health_insurance_premiums_without_medicare_part_b': array([3840, 1068, 2200, ...,    0, 1212,    0]),
 'household_id': array([    1,    19,    55, ..., 89173, 8